# PA2 T4 x2 environment check
Executed Kaggle evidence using the safe no-Torch-reinstall setup.

In [ ]:
import importlib.metadata, subprocess, sys
from pathlib import Path
Path('requirements-kaggle.txt').write_text("# Reproducible candidate stack; keep Kaggle's installed CUDA torch.\n# GPU/model access must still be verified by the gated pilot.\ntransformers==4.51.3\ndatasets==3.6.0\naccelerate==1.6.0\npeft==0.15.2\ntrl==0.18.1\npyyaml==6.0.2\nsentencepiece==0.2.1\nsafetensors==0.5.3\nmatplotlib==3.10.1\npytest==8.3.5\n", encoding='utf-8')
Path('env_check.py').write_text('"""Record the assignment GPU and library environment without changing it."""\nimport argparse\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport subprocess\n\n\nPACKAGES = ("torch", "transformers", "datasets", "trl", "peft", "accelerate")\n\n\ndef collect():\n    import torch\n    try:\n        query = subprocess.run(\n            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],\n            capture_output=True, text=True, check=False)\n        nvidia = query.stdout.strip() or query.stderr.strip()\n    except FileNotFoundError:\n        nvidia = "nvidia-smi unavailable"\n    count = torch.cuda.device_count()\n    return {"nvidia_smi": nvidia, "cuda_device_count": count,\n            "cuda_device_names": [torch.cuda.get_device_name(i) for i in range(count)],\n            "versions": {name: importlib.metadata.version(name) for name in PACKAGES}}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--output", default="outputs/analysis/environment.json")\n    parser.add_argument("--allow-cpu", action="store_true")\n    args = parser.parse_args()\n    result = collect()\n    path = Path(args.output); path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(result, indent=2))\n    if result["cuda_device_count"] < 2 and not args.allow_cpu:\n        raise SystemExit("ERROR: fewer than two GPUs are visible; select Kaggle GPU T4 x2 or pass --allow-cpu.")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=False)
torch_version=importlib.metadata.version('torch')
Path('keep-kaggle-torch.txt').write_text('torch=='+torch_version+'\n')
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-kaggle.txt','-c','keep-kaggle-torch.txt'],check=True)
assert importlib.metadata.version('torch')==torch_version
result=subprocess.run(['nvidia-smi'],text=True,capture_output=True,check=True)
Path('/kaggle/working/nvidia-smi.txt').write_text(result.stdout)
subprocess.run([sys.executable,'env_check.py','--output','/kaggle/working/env_kaggle_t4x2.json'],check=True)
